### Ingestion del archivo "person.json"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

#### Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
name_schema = StructType(fields=[
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True),
])


In [0]:
persons_schema = StructType(fields=[
    StructField("personId", IntegerType(), False),
    StructField("personName", name_schema)
])

In [0]:
persons_df = spark.read \
            .schema(persons_schema) \
            .json(f"{bronze_folder_path}/{v_file_date}/person.json")

In [0]:
persons_df.printSchema()

In [0]:
display(persons_df)

#### Paso 2 - renombrar columnas y agregar columnas nuevas
1. renombrar "personId" a "person_id"
2. Agregar las columnas "ingestion_date" y "environment"
3. Agregar la columna "name" a partir de la concatenacion de "forename" y "surname"

In [0]:
from pyspark.sql.functions import col, concat, current_timestamp, lit

In [0]:
persons_with_columns_df = add_ingestion_date(persons_df) \
                            .withColumnsRenamed({"personId": "person_id"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date)) \
                            .withColumn("name", 
                                        concat(
                                            col("personName.forename"),
                                            lit(" "), 
                                            col("personName.surname")
                                            )
                                        )    

In [0]:
display(persons_with_columns_df)

#### Paso 3 - Eliminar las columnas no necesarias

In [0]:
persons_final_df = persons_with_columns_df.drop(col("personName"))

In [0]:
display(persons_final_df)

#### Paso 4 - Escribir salida en un formato "Parquet"

In [0]:
# persons_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/persons")

In [0]:
# display(spark.read.parquet("abfss://silver@moviehistory22.dfs.core.windows.net/persons"))

In [0]:
# overwrite_partition("movie_silver", "persons","file_date",v_file_date)

In [0]:
# persons_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.persons")
# persons_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.persons")

In [0]:
# hago una validacion con merge

merge_condition = 'tgt.person_id = src.person_id AND tgt.file_date=src.file_date'
merge_delta_lake(persons_final_df, "movie_silver", "persons", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.persons
GROUP BY file_date

In [0]:
%sql
SELECT * FROM movie_silver.persons

In [0]:
dbutils.notebook.exit("Success")